# Presidential speeches -> keyword CSV
### edited to look at immigration-related terms

Makes `speech_keywords.csv` with one row per speech: president, year, title, speech type, and a count for each keyword.

Run this from the repo root. Each file in `speeches/` is one speech stored as JSON, which Python reads as a dictionary with keys like `president`, `date`, `title` and `transcript`.

In [24]:
import glob
import json
import re

import pandas as pd

## Settings

Change these two to change the CSV. The data has no speech-type field, so the type is guessed from the title: the first phrase below that appears in the title wins.

In [25]:
keywords = ["immigration", "immigrants", "migrants", "refugees", "asylum"]
loaded_terms = ['alien','illegal immigrant','illegal immigration','illegal migrant','undocumented']

speech_types = {
    "inaugural": "Inaugural Address",
    "fireside chat": "Fireside Chat",
    "state of the union": "State of the Union",
    "annual message": "State of the Union",
    "annual address": "State of the Union",
    "veto": "Veto Message",
    "proclamation": "Proclamation",
    "debate": "Debate",
    "press conference": "Press Conference",
    "news conference": "Press Conference",
    "briefing": "Press Conference",
    "farewell": "Farewell Address",
    "national convention": "Convention Speech",
    "nomination": "Convention Speech",
    "message": "Message",
    "remarks": "Remarks / Statement",
    "statement": "Remarks / Statement",
    "address": "Address / Speech",
    "speech": "Address / Speech",
}

## Go through every speech

In [26]:
rows = []

for filename in sorted(glob.glob("speeches/*.json")):
    with open(filename, encoding="utf-8") as f:
        speech = json.load(f)

    # speech type: the first phrase that appears in the title
    title = speech["title"].lower()
    speech_type = "Other"
    for phrase, label in speech_types.items():
        if phrase in title:
            speech_type = label
            break

    # split the transcript into lowercase words
    text = speech["transcript"].lower()
    text = re.sub(r"<[^>]+>", " ", text)   # remove HTML tags such as <br />
    words = re.findall(r"[a-z]+", text)

    row = {
        "president": speech["president"],
        "year": int(speech["date"][:4]),
        "title": speech["title"],
        "speech_type": speech_type,
    }
    for keyword in keywords:
        row[keyword] = words.count(keyword)

    rows.append(row)

## Save

In [27]:
df = pd.DataFrame(rows)
df.to_csv("speech_keywords.csv", index=False)
df.head()

,president,year,title,speech_type,immigration,immigrants,migrants,refugees,asylum
0,Abraham Lincoln,1852,"July 6, 1852: Eulogy on Henry Clay",Other,0,0,0,0,0
1,Abraham Lincoln,1854,"October 16, 1854: At Peoria, Illinois",Other,0,0,0,0,0
2,Abraham Lincoln,1858,"June 16, 1858: ""A House Divided"" Speech",Address / Speech,0,0,0,0,0
3,Abraham Lincoln,1860,"February 27, 1860: Cooper Union Address",Address / Speech,0,0,0,0,0
4,Abraham Lincoln,1861,"February 11, 1861: Farewell Address",Farewell Address,0,0,0,0,0


In [28]:
df["speech_type"].value_counts()

speech_type
State of the Union     210
Address / Speech       204
Message                158
Remarks / Statement    139
Other                   87
Inaugural Address       60
Veto Message            44
Press Conference        39
Proclamation            38
Fireside Chat           30
Convention Speech       24
Farewell Address        13
Debate                  13
Name: count, dtype: int64

## Optional: a more sophisticated version

The above cell is the simplest approach. But we can make it more sophisticated.
The cell below is the same loop again, with two additions marked `# NEW`:

- `word_count`, so a count can be turned into a rate per 1,000 words
- word families: `econom*` counts every word that starts with `econom` (*economy*, *economic*, *economics*...)

It saves a second file, `speech_keywords_extended.csv`. Feel free to compare them.

In [29]:
import html                                # NEW

families = ["immigra", "migra"]    # NEW

rows = []

for filename in sorted(glob.glob("speeches/*.json")):
    with open(filename, encoding="utf-8") as f:
        speech = json.load(f)

    # speech type: the first phrase that appears in the title
    title = speech["title"].lower()
    speech_type = "Other"
    for phrase, label in speech_types.items():
        if phrase in title:
            speech_type = label
            break

    # split the transcript into lowercase words
    text = speech["transcript"].lower()
    text = re.sub(r"<[^>]+>", " ", text)   # remove HTML tags such as <br />
    text = html.unescape(text)             # NEW: turn codes like &nbsp; into real characters
    words = re.findall(r"[a-z]+", text)

    row = {
        "president": speech["president"],
        "year": int(speech["date"][:4]),
        "title": speech["title"],
        "speech_type": speech_type,
        "word_count": len(words),          # NEW
    }
    for keyword in keywords:
        row[keyword] = words.count(keyword)

    for term in loaded_terms:              # NEW: Adding loaded_term index
        row[term] = words.count(term)

    family_total = 0
    for family in families:                # NEW: Filtering to only have rows with the keywords we want to analyze
        match_count = len([word for word in words if word.startswith(family)])
        row[family + "*"] = match_count
        family_total += match_count        # Track total matches for this speech


    # Only append the row if at least one family word was found
    if family_total > 0:
        rows.append(row)

df_extended = pd.DataFrame(rows)
df_extended.to_csv("speech_keywords_extended.csv", index=False)
df_extended.head(15)

,president,year,title,speech_type,word_count,immigration,immigrants,migrants,refugees,asylum,alien,illegal immigrant,illegal immigration,illegal migrant,undocumented,immigra*,migra*
0,Abraham Lincoln,1854,"October 16, 1854: At Peoria, Illinois",Other,33240,0,0,0,0,0,0,0,0,0,0,0,2
1,Abraham Lincoln,1861,"December 3, 1861: First Annual Message",State of the Union,6956,1,0,0,0,0,0,0,0,0,0,1,0
2,Abraham Lincoln,1862,"December 1, 1862: Second Annual Message",State of the Union,8294,0,0,0,0,0,0,0,0,0,0,0,2
3,Abraham Lincoln,1863,"December 8, 1863: Third Annual Message",State of the Union,6080,1,0,0,0,0,0,0,0,0,0,1,0
4,Abraham Lincoln,1864,"December 6, 1864: Fourth Annual Message",State of the Union,5804,1,3,0,0,1,0,0,0,0,0,4,0
5,Andrew Jackson,1832,"February 15, 1832: Message Regarding Indian Re...",Message,409,0,0,0,0,0,0,0,0,0,0,0,1
6,Andrew Jackson,1833,"December 3, 1833: Fifth Annual Message to Cong...",State of the Union,7848,0,0,0,0,0,0,0,0,0,0,0,1
7,Andrew Johnson,1865,"December 4, 1865: First Annual Message",State of the Union,9222,0,0,0,0,0,0,0,0,0,0,0,1
8,Andrew Johnson,1866,"March 27, 1866: Veto Message on Civil Rights L...",Veto Message,4428,0,0,0,0,0,0,0,0,0,0,0,1
9,Barack Obama,2008,"August 28, 2008: Acceptance Speech at the Demo...",Convention Speech,4782,1,1,0,0,0,0,0,0,0,0,2,0


### Why add a word count?

Some decades have far more text than others, so raw counts mostly show where the most words are. Dividing by the number of words makes decades comparable. Compare the `war` column with the rate next to it.

In [30]:
by_decade = df_extended.groupby(df_extended["year"] // 10 * 10)[["word_count", "immigrants"]].sum()
by_decade["immigrants_per_1000_words"] = by_decade["immigrants"] / by_decade["word_count"] * 1000
by_decade.sort_values("immigrants", ascending=False).head(10)

,word_count,immigrants,immigrants_per_1000_words
year,,,
2010,183189,51,0.278401
2000,67923,26,0.382786
1900,152831,26,0.170123
1890,139473,18,0.129057
1990,82822,17,0.205259
2020,131659,11,0.083549
1880,109569,9,0.082140
1950,31200,8,0.256410
1910,56327,8,0.142028


### Why add word families?

Here's the exact word next to its family, totalled over all speeches. You can see why this matters.

In [31]:
df_extended[["immigration", "immigra*", "migra*", "refugees", "asylum", "alien", "undocumented"]].sum().sort_values(ascending=False)

immigra*        617
immigration     360
migra*           79
alien            59
undocumented     27
refugees         20
asylum           16
dtype: int64

## Keep in mind

- Debates and press conferences also contain the words of moderators, opponents and reporters.
- A word family can catch words you did not mean, so check what it matches before trusting the column.

### Terms to keep in mind
* nativist/m (1840s anti-immigration sentiments)
* foreigners
* racial/ethnic terms (e.g., Mexican, Chinese, Irish, Somolian, Haitian, etc.)